# Crop-seq (sVAEplus) vs OPS All Fluorescence vs OPS Phase

Compare mean average precision (mAP) distributions across three modalities — the sVAEplus crop-seq embedding, the OPS `cell_dino` All-Fluorescence (no-phase) embedding, and the OPS `cell_dino` Phase-only embedding — on perturbations and protein complexes shared across all three.

## Imports

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["svg.fonttype"] = "none"

FIGURES_DIR = Path("../../output/figure_5")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV paths

All inputs live under `../../data/` (a symlink to the curated central dataset; see README). Notebook paths stay the same on public release — the figshare archive populates the same `data/` layout.

In [ ]:
FIGURE_DATA = Path("../../data/figures/figure_5")

# Crop-seq sVAEplus distinctiveness (variant: std_ntc)
SVAEPLUS_DISTINCTIVENESS_STD_NTC   = FIGURE_DATA / "svaeplus_distinctiveness_std_ntc.csv"

# OPS cell_dino All-Fluorescence (no_phase)
CELL_DINO_NO_PHASE_DISTINCTIVENESS = FIGURE_DATA / "celldino_no_phase_distinctiveness.csv"
CELL_DINO_NO_PHASE_EBI             = FIGURE_DATA / "celldino_no_phase_ebi.csv"

# OPS cell_dino Phase-only
CELL_DINO_PHASE_DISTINCTIVENESS    = FIGURE_DATA / "celldino_phase_only_distinctiveness.csv"
CELL_DINO_PHASE_EBI                = FIGURE_DATA / "celldino_phase_only_ebi.csv"

# Crop-seq EBI mAP
CROPSEQ_EBI                        = FIGURE_DATA / "cropseq_ebi_map.csv"

## Crop-seq vs OPS All Fluorescence vs OPS Phase — shared perturbations / complexes

Restrict each metric to entries shared across all three modalities and compute mean mAP and the per-entry distribution (the latter feeds the violin plot below).

- **Distinctiveness**: 3-way intersection of `perturbation` across crop-seq, OPS All-Fluorescence, and OPS Phase.
- **EBI**: 3-way intersection of `complex_num` across the same three modalities.

In [ ]:
cs_distinct = pd.read_csv(SVAEPLUS_DISTINCTIVENESS_STD_NTC)
ao_distinct = pd.read_csv(CELL_DINO_NO_PHASE_DISTINCTIVENESS)
p_distinct  = pd.read_csv(CELL_DINO_PHASE_DISTINCTIVENESS)

cs_ebi = pd.read_csv(CROPSEQ_EBI)
ao_ebi = pd.read_csv(CELL_DINO_NO_PHASE_EBI)
p_ebi  = pd.read_csv(CELL_DINO_PHASE_EBI)

rows = []
distributions = {}  # (metric, modality) -> per-entry mAP array for the violin plot

# distinctiveness: 3-way shared on "perturbation"
common = set(cs_distinct["perturbation"]) & set(ao_distinct["perturbation"]) & set(p_distinct["perturbation"])
cs_sub = cs_distinct[cs_distinct["perturbation"].isin(common)]
ao_sub = ao_distinct[ao_distinct["perturbation"].isin(common)]
p_sub  = p_distinct[p_distinct["perturbation"].isin(common)]
distributions[("distinctiveness", "crop-seq")]  = cs_sub["mean_average_precision"].values
distributions[("distinctiveness", "all_fluor")] = ao_sub["mean_average_precision"].values
distributions[("distinctiveness", "phase")]     = p_sub["mean_average_precision"].values
rows.append({
    "metric": "distinctiveness", "n_shared": len(common),
    "crop-seq mean":  cs_sub["mean_average_precision"].mean(),
    "all_fluor mean": ao_sub["mean_average_precision"].mean(),
    "phase mean":     p_sub["mean_average_precision"].mean(),
})

# ebi: 3-way shared on complex_num
common = set(cs_ebi["complex_num"]) & set(ao_ebi["complex_num"]) & set(p_ebi["complex_num"])
cs_sub = cs_ebi[cs_ebi["complex_num"].isin(common)]
ao_sub = ao_ebi[ao_ebi["complex_num"].isin(common)]
p_sub  = p_ebi[p_ebi["complex_num"].isin(common)]
distributions[("ebi", "crop-seq")]  = cs_sub["mean_average_precision"].values
distributions[("ebi", "all_fluor")] = ao_sub["mean_average_precision"].values
distributions[("ebi", "phase")]     = p_sub["mean_average_precision"].values
rows.append({
    "metric": "ebi", "n_shared": len(common),
    "crop-seq mean":  cs_sub["mean_average_precision"].mean(),
    "all_fluor mean": ao_sub["mean_average_precision"].mean(),
    "phase mean":     p_sub["mean_average_precision"].mean(),
})

cmp_df = pd.DataFrame(rows)
cmp_df

## mAP distributions — paper figure

Single-panel violin comparison of per-entry mAP distributions across the three modalities (crop-seq, OPS All Fluorescence, OPS Phase) for both distinctiveness and EBI complex consistency, restricted to entries shared across all three modalities. Saved as SVG for the paper.

In [ ]:
import matplotlib.patches as mpatches

metrics_order   = ["distinctiveness", "ebi"]
modality_order  = ["crop-seq", "all_fluor", "phase"]
modality_labels = {"crop-seq": "crop-seq", "all_fluor": "OPS All Fluorescence", "phase": "OPS Phase"}
COLORS          = {"crop-seq": "#4C72B0", "all_fluor": "#DD8452", "phase": "#55A868"}

x = list(range(len(metrics_order)))
width = 0.27

fig, ax = plt.subplots(figsize=(7, 5))

positions, box_data, box_colors = [], [], []
for i, m in enumerate(metrics_order):
    for j, mod in enumerate(modality_order):
        positions.append(i + (j - 1) * width)
        box_data.append(distributions[(m, mod)])
        box_colors.append(COLORS[mod])

vp = ax.violinplot(box_data, positions=positions, widths=width * 0.9,
                   showmedians=True, showextrema=True)
for body, color in zip(vp["bodies"], box_colors):
    body.set_facecolor(color)
    body.set_edgecolor("black")
    body.set_linewidth(0.8)
    body.set_alpha(1.0)
for partname in ("cbars", "cmins", "cmaxes", "cmedians"):
    if partname in vp:
        vp[partname].set_color("black")
        vp[partname].set_linewidth(0.8)

ns = [cmp_df.loc[cmp_df["metric"] == m, "n_shared"].values[0] for m in metrics_order]
ax.set_xticks(x)
ax.set_xticklabels([f"{m}\n(n={n})" for m, n in zip(metrics_order, ns)], fontsize=9)
ax.set_ylabel("mAP")
ax.set_ylim(-0.05, 1.05)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
ax.grid(axis="y", linewidth=0.5, alpha=0.5)

handles = [mpatches.Patch(facecolor=COLORS[m], edgecolor="black", linewidth=0.8, label=modality_labels[m]) for m in modality_order]
ax.legend(handles=handles, loc="upper left")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "cropseq_vs_OPS_map.svg", bbox_inches="tight")
plt.show()